# LeNet

> Lenet-based CNN models


In [ ]:
#| default_exp models.vision.lenets

In [ ]:
#| hide
from nbdev.showdoc import * 

In [ ]:
#| export
from fastcore.utils import * 
from torch import nn
import torch.nn.functional as F
import torch
from typing import Tuple, List, Dict, Any

## MNIST models

In [ ]:
#| export
class MNISTCNN(nn.Module):
    def __init__(self, 
                 img_size: Tuple[int, int, int]= (1, 28, 28), 
                 hidden_dim: int=512,
                 num_classes: int= 10):
        
        super(MNISTCNN, self).__init__()
        in_channels, h, w = img_size    
        self.hidden_dim = hidden_dim

        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=1, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=5, stride=1, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        final_size = h // 2 // 2
        self.fc1 = nn.Linear(64 * final_size * final_size, hidden_dim)
        self.fc2 = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        out = self.conv1(x)
        out = self.conv2(out)
        out = torch.flatten(out, 1)
        out = self.fc1(out)
        out = self.fc2(out)
        return out

    

In [ ]:
#| hide
inp = torch.randn(1, 1, 28, 28)  # Example input
model = MNISTCNN()
output = model(inp)
print(output.shape)  # Should be [1, 10]

model.hidden_dim

torch.Size([1, 10])


512

## CIFAR10 models

In [ ]:
#| export
import torch
import torch.nn as nn
import numpy as np
import random

torch.manual_seed(42)  
np.random.seed(42)
random.seed(42)

class CIFAR10CNN(nn.Module):
    def __init__(self, 
                 img_size: Tuple[int, int, int]= (3, 32, 32), 
                 hidden_dim: int=512,
                 num_classes: int= 10):
        
        super(CIFAR10CNN, self).__init__()
        in_channels, h, w = img_size
        self.hidden_dim = hidden_dim

        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        final_size = h // 2 // 2 // 2  # 3 pool layers divide the size by 2 three times
        self.fc1 = nn.Sequential(nn.Linear(128 * final_size * final_size, hidden_dim),
                                 nn.ReLU(inplace=True)
                                 )
        
        self.fc2 = nn.Linear(hidden_dim, num_classes)
        # self.fc2 = nn.Sequential(
        #     nn.Dropout(0.5),
        #     nn.Linear(hidden_dim, num_classes),
        # )
        

    def forward(self, x):
        out = self.conv1(x)
        out = self.conv2(out)
        out = self.conv3(out)
        out = torch.flatten(out, 1)
        out = self.fc1(out)
        out = self.fc2(out)
        return out



In [ ]:
#| hide
model = CIFAR10CNN(img_size=(3, 32, 32), hidden_dim=512, num_classes=10)
inp = torch.randn(1, 3, 32, 32)
out = model(inp)
print(out.shape)  

model = CIFAR10CNN(img_size=(1, 28, 28), hidden_dim=512, num_classes=10)
inp = torch.randn(1, 1, 28, 28)
out = model(inp)
print(out.shape)

model.hidden_dim

torch.Size([1, 10])
torch.Size([1, 10])


512

## FedAvg net

In [ ]:
#| export
class FedAvgCNN(nn.Module):
    def __init__(self, 
                 img_size: Tuple[int, int, int]= (3, 32, 32), 
                 hidden_dim: int=512,
                 num_classes: int= 10):
        super().__init__()

        in_channels, h, w = img_size
        self.hidden_dim = hidden_dim
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels,
                      32,
                      kernel_size=5,
                      padding=0,
                      stride=1,
                      bias=True),
            nn.ReLU(inplace=True), 
            nn.MaxPool2d(kernel_size=(2, 2))
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32,
                      64,
                      kernel_size=5,
                      padding=0,
                      stride=1,
                      bias=True),
            nn.ReLU(inplace=True), 
            nn.MaxPool2d(kernel_size=(2, 2))
        )

        def compute_dim(h, w):
            h = (h - 4) // 2  # After conv1 and pool
            w = (w - 4) // 2
            h = (h - 4) // 2  # After conv2 and pool
            w = (w - 4) // 2
            return 64 * h * w


        dim = compute_dim(h, w)
        self.fc1 = nn.Sequential(
            nn.Linear(dim, hidden_dim), 
            nn.ReLU(inplace=True)
        )
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        out = self.conv1(x)
        out = self.conv2(out)
        out = torch.flatten(out, 1)
        out = self.fc1(out)
        out = self.fc2(out)
        return out

In [ ]:
#| hide
model = FedAvgCNN(img_size=(1, 28, 28), hidden_dim=512, num_classes=10)
inp = torch.randn(1, 1, 28, 28)
out = model(inp)
print(out.shape)  # Should be [1, 10]

torch.Size([1, 10])


In [ ]:
img_size= 32
final_size = img_size // 2 // 3 #// 2  # 3 pool layers divide the size by 2 three times
64 * final_size * final_size

1600

### Unified API for all Lenet-based models

In [ ]:
#| export
from fedai.models.vision.registery import register_model
from typing import Literal

In [ ]:
#| export
@register_model(
    name='lenet',
    variants=['mnist', 'cifar10', 'fedavg']
)
def make_lenet(
    variant: Literal['mnist', 'cifar10', 'fedavg'] = 'fedavg',
    pretrained: bool = False, # Note: pretrained is not implemented for these simple models, but we include it in the signature for API consistency
    num_classes: int = 10,
    img_size: Tuple[int, int, int] = (3, 32, 32),
    hidden_dim: int = 512,
) -> nn.Module:
    
    model_dict = {
        'fedavg': FedAvgCNN,
        'mnist': MNISTCNN,
        'cifar10': CIFAR10CNN
    }

    if variant not in model_dict:
        raise ValueError(f"Variant '{variant}' not recognized. Available variants: {list(model_dict.keys())}")
    
    model_cls = model_dict[variant]
    model = model_cls(img_size=img_size, hidden_dim=hidden_dim, num_classes=num_classes)
    return model


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()